In [ ]:
import sys

In [ ]:
import sys
print(sys.executable)


In [ ]:
!pip install scikit-learn

In [ ]:
import sys
!{sys.executable} -m pip install --upgrade pip
!{sys.executable} -m pip install nltk
import nltk
nltk.download('stopwords')

In [ ]:
import sklearn


In [ ]:
# --- Environment Setup for VS Code ---

import pandas as pd
import numpy as np
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import nltk

# Download NLTK stopwords (only the first time)
nltk.download('stopwords')
from nltk.corpus import stopwords
stopwords_pt = stopwords.words('portuguese')




#Sprint 2

In [ ]:
# carregar o dataset 

parquet_path = "../../data/bronze/cision_news_20251110.parquet"

sample = pd.read_parquet(parquet_path, engine="pyarrow")

sample.head()


In [ ]:
sample.head(10)

In [ ]:
print(sample.shape)
print(sample.columns)
display(sample.head())
display(sample.dtypes)
print(sample.isna().sum())
print("Duplicados por corpo de noticia:", sample.duplicated(subset=["noticia"]).sum()) #mas têm valores unitarios dif


só a coluna "categoria" é que tem nulos

In [ ]:
# Datas e valores numéricos
sample["data_publicacao"] = pd.to_datetime(sample["data_publicacao"], errors="coerce")
sample["valor_publicitario"] = pd.to_numeric(sample["valor_publicitario"], errors="coerce")

# Normalizar texto das colunas relevantes
def norm_txt(s):
    s = str(s).lower()
    s = re.sub(r"[\:\;\,\.\!\?\(\)\[\]\-\–\—\"\']", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

#vamos ver com o corpo da noticia
sample["noticia"] = sample["noticia"].map(norm_txt)


##TF-IDF (melhor opção de todas)

In [ ]:
#4. Vetorização TF-IDF + matriz de similaridade
vectorizer = TfidfVectorizer(stop_words=list(stopwords_pt), ngram_range=(1,2), max_features=30000, min_df=3)
X = vectorizer.fit_transform(sample["noticia"])

# Similaridade (mais baixo aumenta grupos)
from sklearn.neighbors import NearestNeighbors
from scipy.sparse.csgraph import connected_components

k = 25           # começa com 25
threshold = 0.76 # o mesmo que já usavas

nbrs = NearestNeighbors(n_neighbors=k, metric="cosine", n_jobs=-1)
nbrs.fit(X)

# grafo de distâncias (esparso) -> similaridade
G = nbrs.kneighbors_graph(mode="distance").tocsr()
G.data = 1.0 - G.data
G.data[G.data < threshold] = 0.0
G.eliminate_zeros()

# grupos por componentes conexos (labels = grupo de cada linha)
B = G.copy(); B.data[:] = 1.0
n_comp, labels = connected_components(csgraph=B, directed=False, return_labels=True)


In [ ]:
#5. Agrupamento por similaridade
sample = sample.reset_index(drop=True)
sample["grupo_id"] = pd.Series(labels).astype("Int64")


In [ ]:
# para contagens
from collections import Counter

sizes = Counter(sample["grupo_id"].value_counts().tolist())
print("Distribuição de tamanhos de grupo:", sizes)
print("Grupos totais:", sample["grupo_id"].nunique())
print("Grupos com 1 notícia:", (sample["grupo_id"].value_counts()==1).sum())



In [ ]:
# Tabela com número de notícias por grupo
contagem_por_grupo = (
    sample.groupby("grupo_id")
          .size()
          .reset_index(name="num_noticias")
)

# Lista com os ids dos grupos que têm exatamente x notícias p.e.
grupos_x = contagem_por_grupo.loc[
    contagem_por_grupo["num_noticias"] ==47,"grupo_id"
].tolist()

print("Grupos com x notícias:", grupos_x)
print("Total de grupos com x notícias:", len(grupos_x))

In [ ]:
#6. Escolher o líder por valor publicitário

def lider_valor_publicitario(df, grupo, col_valor="valor_publicitario"):
    # apenas retorna o índice da notícia com maior valor_publicitario dentro do grupo
    idx_validos = [int(i) for i in grupo if 0 <= int(i) < len(df)]
    return int(df.loc[idx_validos, col_valor].idxmax())

leaders = (sample.groupby("grupo_id")["valor_publicitario"].idxmax().dropna().astype(int).tolist())
sample["is_lider"] = False
sample.loc[leaders, "is_lider"] = True

In [ ]:
#dos 4 grupos que tem 15 noticias
gid = grupos_x[0] #0 é o 1 grupo, 1 o segundo e etc
exemplo_4 = sample.loc[sample["grupo_id"] == gid, ["grupo_id","is_lider","noticia","data_publicacao", "valor_publicitario"]].head(50)
display(exemplo_4)


In [ ]:
ver_carac = sample[sample["grupo_id"] == 391 ]

display(ver_carac[["is_lider", "noticia", "tipo_meio", "valor_publicitario"]])


In [ ]:
# Inspecionar um grupo grande
gid_ex = sample["grupo_id"].value_counts().idxmax()
display(sample[sample["grupo_id"] == gid_ex][["id", "is_lider", "noticia", "valor_publicitario"]].head(50))
